In [4]:
!pip install adapters -q

In [5]:
import pandas as pd
from transformers import TrainingArguments, EarlyStoppingCallback, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
import adapters
from adapters import AutoAdapterModel, SeqBnConfig, AdapterTrainer
#SeqBnConfig = Pfeiffer

In [6]:
drive.mount('/content/drive')

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer"
os.makedirs(output_dir, exist_ok=True)


Mounted at /content/drive


Loading the dataset

In [7]:
scotbess_df = pd.read_csv(SPLIT_PATH)

In [8]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)

Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [9]:
mlb = joblib.load(MLB_PATH)

print("Number of labels:", len(mlb.classes_))
print(mlb.classes_)

Number of labels: 20
['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']


In [10]:
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return json.loads(value)

for df in [scotbess_df_train, scotbess_df_val, scotbess_df_test]:
    df["label_list"] = df["labels"].apply(parse_labels)


In [11]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])


In [12]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [13]:
scotbess_X_train = scotbess_df_train["final_masked_text"].fillna("").astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].fillna("").astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].fillna("").astype(str)

In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
# for DistilBERT max token length is 512 - truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(scotbess_X_train)
dev_enc = tokenize(scotbess_X_val)
test_enc = tokenize(scotbess_X_test)

In [16]:
y_train_bin = scotbess_y_train.astype(np.float32)
y_dev_bin = scotbess_y_val.astype(np.float32)
y_test_bin = scotbess_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)


['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1340, 20)


In [17]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [18]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int) #default for now

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

In [20]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (1340, 20)
Val labels: (165, 20)
Test labels: (170, 20)
Number of labels: 20


**Training function**

In [21]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)

    if measure_vram:
        reset_cuda_peak_memory()

    model = AutoAdapterModel.from_pretrained(config["base_model"])
    #remove unnecessary default/pretraining head as a check revealed it is present
    if "default" in model.heads:
        model.delete_head("default")

    #the adapters' library developers wrote on github that multilabel classification head is supported out of the box and can be implemented like this
    model.add_classification_head("scotbess", num_labels=len(mlb.classes_),  multilabel=True, id2label=id2label)

    adapter_config = SeqBnConfig(reduction_factor=config["reduction_factor"])
    model.add_adapter("scotbess", config=adapter_config, set_active=True)
    model.train_adapter("scotbess")

    #check whether the adapters are working
    print("Active adapters:", model.active_adapters)
    print(model.adapter_summary())
    #check the classification head
    print("Trainable classification/head parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad and ("head" in name.lower() or "classifier" in name.lower() or "classification" in name.lower()):
           print(name, param.numel())

    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")
    #adapter trainer, as recommended in the library's documentation
    trainer = AdapterTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])

    #check
    print("Active adapters after trainer creation:", trainer.model.active_adapters)

    #for resuming if something goes wrong//collab's runtime gets disconnected
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


    sync_cuda()
    train_start = time.perf_counter()
    #includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "Scot-BESS",
        "method": "pfeiffer",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        #single forward pass, gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        #derive predictions, needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(output_dir, f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

            #for saving the adapter
            adapter_save_path = os.path.join(config["output_dir"], "final_Pfeiffer_adapter_with_head")
            trainer.model.save_adapter(adapter_save_path, "scotbess", with_head=True)
            result["saved_adapter_path"] = adapter_save_path
            print(f"Adapter + head saved to {adapter_save_path}")


    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

**Hyperparameter search**

In [22]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",

    "max_length": 512,
    "num_train_epochs": 10,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,

    "reduction_factor": 8}  #the default one is 16, but since DistilBERT is already a smaller model I will go with 8, this is also the choice of  Razuvayevskaya et al. (2024)

#small search on the most relevant hyperparameters
learning_rates = [1e-4, 2e-4, 5e-4] #1e-4 is recommmended in the adapters library documentation, 2e-4 is used by Razuvayevskaya et al. (2024)
batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipping already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running DistilBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed

#the warning about adapters can be ignored, based on the check the adapters work
#the warining about early stopping can also be ignored, it works

search_results_df

Running DistilBERT: lr=0.0001, batch_size=8


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.583100,0.471844,0.581369,0.426507
2,0.468700,0.436622,0.573201,0.398382
3,0.408200,0.392943,0.658480,0.525371
4,0.363300,0.362087,0.712681,0.584280
5,0.336300,0.351383,0.704367,0.575996
6,0.318700,0.337534,0.738058,0.639022
7,0.304200,0.332817,0.743631,0.633801
8,0.294400,0.331709,0.750779,0.645139
9,0.289900,0.324583,0.751439,0.653247
10,0.284600,0.324364,0.752371,0.653804


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


...

Running DistilBERT: lr=0.0005, batch_size=16


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.549700,0.468189,0.543030,0.370090
2,0.414600,0.373414,0.689891,0.545309
3,0.344200,0.341325,0.724294,0.593516
4,0.299500,0.318570,0.771386,0.680591
5,0.271600,0.300177,0.777893,0.709363
6,0.247700,0.294981,0.794729,0.728128
7,0.224000,0.285825,0.791345,0.712282
8,0.204900,0.288343,0.791601,0.710560
9,0.192300,0.277092,0.798777,0.736398
10,0.181100,0.278523,0.804966,0.740434


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.757201,10.0,155.464007,0.920213,None,1495892,67858772,0.757201,0.806824,156.384221
1,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.740434,10.0,149.483862,0.834525,None,1495892,67858772,0.740434,0.804966,150.318386
2,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0002,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.702385,10.0,155.642236,0.830803,None,1495892,67858772,0.702385,0.781314,156.473039
3,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0002,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.675376,10.0,149.632931,0.783205,None,1495892,67858772,0.675376,0.766374,150.416136
4,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0001,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.653804,10.0,195.554205,0.807635,None,1495892,67858772,0.653804,0.752371,196.361840
5,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0001,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.607666,10.0,152.910923,0.792403,None,1495892,67858772,0.607666,0.729022,153.703326


**Best configuration**

In [23]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best batch size: 8
Best validation macro-F1: 0.7572006135253607
Best checkpoint: /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/search/lr_0.0005_bs_8/checkpoint-1512


In [24]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [25]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

In [26]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "SCOTBESS_DistilBERT_Pfeiffer_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"SCOTBESS_DistilBERT_Pfeiffer_test_seed_{seed}"))

    # skips already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    #saves incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.0005, batch_size=8


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.529100,0.434287,0.617312,0.442173
2,0.388100,0.351363,0.730584,0.593487
3,0.324000,0.329028,0.732026,0.613703
4,0.281300,0.306754,0.772414,0.676537
5,0.248400,0.289926,0.791795,0.719559
6,0.216700,0.286043,0.801196,0.748871
7,0.190700,0.278469,0.803181,0.741849
8,0.168700,0.279572,0.802015,0.738836
9,0.152500,0.274335,0.808875,0.760922
10,0.139800,0.275270,0.802051,0.739240


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/classification_report_seed_0.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/test_predictions_seed_0.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/test/SCOTBESS_DistilBERT_Pfeiffer_test_seed_0/final_Pfeiffer_adapter_with_head
Final run: seed=1, lr=0.0005, batch_size=8


...

Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.534600,0.442026,0.585337,0.397771
2,0.389100,0.349781,0.715556,0.570606
3,0.321100,0.321330,0.750135,0.642828
4,0.277100,0.300794,0.779138,0.688334
5,0.245000,0.284414,0.794073,0.730399
6,0.211300,0.281770,0.806467,0.754967
7,0.184600,0.268421,0.814150,0.767058
8,0.164500,0.268428,0.810811,0.750461
9,0.147600,0.268009,0.815377,0.770201
10,0.135000,0.267454,0.814664,0.764784


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/test_predictions_seed_2.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Pfeiffer/test/SCOTBESS_DistilBERT_Pfeiffer_test_seed_2/final_Pfeiffer_adapter_with_head


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,saved_adapter_path,total_measured_time_sec
0,distilbert-base-uncased,Scot-BESS,pfeiffer,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.760922,10.0,...,0.808875,0.874575,5.144561,0.748624,0.799403,5.900000,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,157.337917
1,distilbert-base-uncased,Scot-BESS,pfeiffer,1,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.766116,10.0,...,0.811463,0.879789,5.175226,0.763503,0.811695,5.952941,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,156.434387
2,distilbert-base-uncased,Scot-BESS,pfeiffer,2,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.770201,10.0,...,0.815377,0.938162,5.518602,0.751520,0.804563,5.941176,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,156.718224


In [27]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params",
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "SCOTBESS_DistilBERT_Pfeiffer_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.754549,0.805220,5.931373,5.917647,155.090772,0.841895,0.897509,5.279463,0.934281,10.0,1495892.0,67858772.0,156.830176
std,0.007888,0.006172,0.027799,0.000000,0.479744,0.034225,0.035303,0.207667,0.001558,0.0,0.0,0.0,0.462051
